# Лабораторная работа № 5
## Декораторы, генераторы и itertools. Базовые паттерны проектирования

**Среда выполнения:** Google Colab + GitLab

**Прикладная задача:** «Конвейер обработки событий мониторинга» — собрать подсистему, которая принимает поток событий, фильтрует и обогащает его ленивыми итераторами, измеряет и кэширует операции декораторами и собирает архитектуру из пяти базовых паттернов: Фасад, Наблюдатель, Фабрика, Стратегия, Синглтон

**Автор:** <ФИО>

**Группа:** <номер группы>

**Дата:** <дата>


---
## Часть I. Теория

### 1. Функции как объекты первого класса

В Python функция — экземпляр типа `object`: её можно присвоить переменной, положить в словарь или список, передать аргументом и вернуть из другой функции. Именно это делает декораторы возможными: декоратор — обычная функция, принимающая функцию и возвращающая функцию.

Определение функции исполняется в момент `def`, поэтому имя функции становится доступным сразу после объявления, а её атрибуты (`__name__`, `__doc__`, `__closure__`) можно читать и менять. Граница между «кодом» и «данными» в Python размыта: функция — значение, а вызов — операция над значением.

*   `f = abs` — ссылка на объект функции, вызов `f(-3)`.
*   `ops = {"sum": lambda a, b: a + b}` — функция как значение в реестре.
*   `def apply(fn, xs): return [fn(x) for x in xs]` — функция-аргумент.
*   Вложенная функция замыкается на переменные внешней области видимости (LEGB).

```python
def bold(text: str) -> str:
    return f"<b>{text}</b>"

def italic(text: str) -> str:
    return f"<i>{text}</i>"

renderers = [bold, italic]
print([r("ok") for r in renderers])
print(type(bold), bold.__name__)
```

**Вывод.** Декоратор — не особая синтаксическая конструкция, а следствие того, что функция является объектом первого класса.

### 1.1. Базовая конструкция декоратора

Запись `@deco` над определением функции эквивалентна `func = deco(func)`: интерпретатор вызывает декоратор один раз — в момент определения функции, а не в момент вызова. Тело декоратора выполняется однократно, тело обёртки — при каждом вызове decorateемой функции.

Обёртка обязана принимать произвольную сигнатуру оригинала (`*args, **kwargs`) и возвращать его результат, иначе декорируемая функция потеряет контракт. Если декоратор не возвращает обёртку, а возвращает `None`, оригинал уничтожается.

Рис. 1. Декоратор применяется один раз при определении, обёртка — при каждом вызове

```mermaid
flowchart LR
    A["определение func"] --> B["func = logged(func)"]
    B --> C["в func лежит wrapper"]
    D["вызов func(args)"] --> E["сначала тело wrapper"]
    E --> F["затем оригинал и результат"]
```

```python
def logged(func):
    def wrapper(*args, **kwargs):
        print(f"call {func.__name__}{args}")
        return func(*args, **kwargs)
    return wrapper

@logged
def latency_p95(values):
    return sorted(values)[int(len(values) * 0.95) - 1]

print(latency_p95([12.0, 8.5, 40.2, 3.1, 15.7]))
```

**Вывод.** Декоратор меняет поведение, не трогая исходный код функции, — это наименее инвазивный способ расширения функциональности.

### 1.2. `@wraps` и сохранение метаданных

Без `functools.wraps` обёртка перекрывает имя, docstring и сигнатуру оригинала: `func.__name__` начинает возвращать `wrapper`, а докстринг и аннотации теряются. Это ломает рефлексию, `help()`, сериализацию вызовов и тесты, проверяющие сигнатуру.

`@wraps(func)` копирует метаданные `__module__`, `__name__`, `__qualname__`, `__doc__`, `__dict__` и добавляет атрибут `__wrapped__`, через который доступен оригинал — именно его используют `inspect.signature` и отладчики.



| Свойство | Без `@wraps` | С `@wraps` |
| --- | --- | --- |
| `__name__` | `wrapper` | имя оригинала |
| `__doc__` | docstring обёртки | docstring оригинала |
| `inspect.signature` | сигнатура обёртки | сигнатура оригинала |
| `__wrapped__` | нет | ссылка на оригинал |

```python
from functools import wraps

def counted(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@counted
def total_latency(events):
    """Суммарная задержка по потоку событий."""
    return sum(e.latency_ms for e in events)

total_latency([])
print(total_latency.__name__, total_latency.__doc__.strip(), total_latency.calls)
```

**Вывод.** Правило хорошего тона: любой декоратор функции обязан использовать `@wraps`, иначе декорирование портит публичный контракт.

### 1.3. Декораторы с аргументами

Аргументы у `@deco(...)` читаются в момент определения функции, поэтому такой декоратор — функция-фабрика: она принимает параметры и возвращает настоящий декоратор. Получается три вложенных уровня: параметры → декоратор → обёртка.

Параметры захватываются замыканием и живут всё время существования функции. Изменяемое состояние в этом захвате — источник утечек и гонок при многопоточности, поэтому счётчики и кэши в параметризованных декораторах делают либо неизменяемыми, либо защищёнными `threading.Lock`.

```python
from functools import wraps

def timed(threshold_ms: float = 100.0):
    def deco(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            t0 = time.perf_counter()
            result = func(*args, **kwargs)
            took = (time.perf_counter() - t0) * 1000
            if took > threshold_ms:
                print(f"SLOW {func.__name__}: {took:.3f} мс")
            return result
        return wrapper
    return deco

@timed(threshold_ms=0.0)
def scan(events):
    return sum(1 for e in events if e.level == 'error')

_ = scan(EVENTS)
```

**Вывод.** Декоратор без аргументов — `deco(func)`, с аргументами — `deco(params)(func)`; разница только в количестве уровней вложенности.

### 1.4. Порядок применения, условные и классовые декораторы

Несколько декораторов применяются снизу вверх: ближний к функции становится внешним по отношению к оригиналу, но внутренним по отношению к обёртке. При вызове сначала отработает верхний декоратор, затем нижний, затем тело функции — то есть оборачивание и раскручивание идут в разных направлениях.

Условный декоратор принимает решение об обёртывании во время импорта: если условие не выполнено, возвращается исходная функция. Класс-декоратор использует `__init__` для захвата оригинала и `__call__` для перехвата вызова, что удобно, когда у декоратора есть настраиваемое состояние и его нужно переиспользовать.

```python
def tag(name):
    def deco(func):
        def wrapper(*a, **kw):
            return f"<{name}>{func(*a, **kw)}</{name}>"
        return wrapper
    return deco

@tag("div")
@tag("b")
def text():
    return "hello"

print(text())   # <div><b>hello</b></div>

class Repeat:
    def __init__(self, func, times=2):
        self.func, self.times = func, times
    def __call__(self, *a, **kw):
        return self.func(*a, **kw) * self.times

@Repeat
def mark():
    return "!"

print(mark())
```

**Вывод.** `@a @b f` — это `a(b(f))`: читать декораторы нужно снизу вверх, а исполнять обёртки — сверху вниз.

### 1.5. Декораторы стандартной библиотеки

`functools` покрывает большинство производственных нужд: `lru_cache` и `cache` мемоизируют чистые функции, `singledispatch` выбирает реализацию по типу первого аргумента, `partial` фиксирует часть аргументов, `cached_property` вычисляет атрибут один раз на экземпляр.

Отдельная группа — декораторы, регистрирующие сущность: `@app.route` во Flask, `@pytest.fixture`, `@abstractmethod`, `@property`, `@dataclass`. Они не меняют поведение вызова, а вносят функцию в реестр или меняют способ её объявления — это третья классическая цель декораторов после функциональных дополнений и санирования данных.



| Декоратор | Назначение | Типичная ошибка |
| --- | --- | --- |
| `lru_cache(maxsize=n)` | мемоизация чистых функций | немутабельные аргументы вне кэша |
| `singledispatch` | перегрузка по типу аргумента | регистрация без `@func.register` |
| `cached_property` | ленивый атрибут экземпляра | зависимость от изменяемого состояния |
| `property` | контролируемый доступ к атрибуту | валидация только в геттере |
| `dataclass` | генерация `__init__`, `__repr__`, `__eq__` | `mutable default` без `field(default_factory=…)` |

```python
from functools import lru_cache, singledispatch

@lru_cache(maxsize=None)
def bucket(latency_ms: int) -> str:
    return 'fast' if latency_ms < 20 else ('mid' if latency_ms < 60 else 'slow')

print(bucket(15), bucket(45), bucket(300), bucket.cache_info())

@singledispatch
def to_ms(value):
    raise TypeError(type(value))

@to_ms.register(float)
def _(value):
    return round(value * 1000, 2)

@to_ms.register(int)
def _(value):
    return float(value)

print(to_ms(0.25), to_ms(25))
```

**Вывод.** Прежде чем писать свой декоратор, проверьте `functools`: мемоизация, диспетчирование и кэш свойств уже решены стандартно.

### 2. Итераторы: iterable, iterator, итерация

Итерируемый объект — тот, у которого есть `__iter__`, возвращающий итератор, либо `__getitem__`, принимающий целые индексы от нуля. Итератор — объект с `__next__`, который при исчерпании выбрасывает `StopIteration`.

Ключевое отличие от контейнера: итератор помнит только позицию и не знает длины. Итератор можно израсходовать — повторный проход по нему даёт пустой результат, и это самый частый источник ошибок в конвейерах. `list(iterable)` материализует последовательность и делает её многопроходной ценой памяти.

```python
it = iter(['auth', 'cart', 'billing'])
print(next(it), next(it))
try:
    print(next(it))
    print(next(it))
except StopIteration:
    print('StopIteration: итератор исчерпан')

gen = (e.service for e in EVENTS)
print(sum(1 for _ in gen), sum(1 for _ in gen))   # второй проход пуст
```

**Вывод.** Протокол итерации — единый интерфейс перебора: цикл `for`, распаковка, `list()` и `itertools` работают с любым объектом, его соблюдающим.

### 2.1. Генераторы и `yield`

Функция, содержащая `yield`, становится генератором: её вызов не исполняет тело, а возвращает объект-генератор. При каждом `next` исполнение доходит до `yield`, замораживается с сохраном локальных переменных и продолжается с того же места.

Генератор — это ленивый конвейер: он обрабатывает элементы по одному, поэтому память не зависит от объёма входа. Для магистратуры важна вторая сторона — время: цепочка генераторов не читается «из начала», и повторный проход требует пересчёта всего.

Методы `send`, `throw` и `close` превращают генератор в корутину: `send` передаёт значение в выражение `yield`, `close` выбрасывает `GeneratorExit` внутри генератора.

```python
def errors_only(events):
    for e in events:
        if e.level == 'error':
            yield e

def enriched(events):
    for e in events:
        yield (e.service, round(e.latency_ms, 1), e.ts.hour)

pipe = enriched(errors_only(EVENTS))
print(type(pipe))
print(next(pipe), next(pipe))
print(sum(1 for _ in pipe), 'осталось после двух next')
```

**Вывод.** `yield` даёт инкрементальный алгоритм без промежуточных списков: конвейер из генераторов читает ровно столько, сколько нужно.

### 3. Модуль itertools: алгебра итераторов

`itertools` стандартизирует быстрый и экономный по памяти набор инструментов над итераторами. Все функции возвращают итераторы, ничего не материализуют и комбинируются друг с другом — это и есть «алгебра итераторов»: сложные выборки собираются из простых операторов.

Инструменты делятся на три группы: бесконечные (`count`, `cycle`, `repeat`), конечные (`chain`, `islice`, `dropwhile`, `takewhile`, `filterfalse`, `accumulate`, `compress`, `groupby`, `starmap`) и комбинаторные (`product`, `permutations`, `combinations`, `combinations_with_replacement`).

Рис. 2. Три семейства итераторов и общая точка сборки — ленивый конвейер

```mermaid
flowchart LR
    A["itertools"] --> B["бесконечные: count, cycle"]
    A --> C["конечные: chain, islice, groupby"]
    A --> D["комбинаторные: product, combinations"]
    B --> F["ленивый конвейер: O(1) памяти"]
    C --> F
    D --> F
```



| Группа | Инструмент | Что даёт | Цена |
| --- | --- | --- | --- |
| бесконечные | `count(a, step)` | ряд значений | только `islice` остановит |
| бесконечные | `cycle(iterable)` | зацикленный повтор | копия входа в памяти |
| конечные | `chain(*its)` | одна последовательность из многих | O(1) памяти |
| конечные | `islice(it, stop)` | срез без индексов | пропуск `start` элементов |
| конечные | `groupby(it, key)` | группы подряд идущих | требует сортировки |
| комбинаторные | `combinations(it, r)` | сочетания | C(n, r) элементов |

```python
from itertools import count, cycle, islice, chain

print(list(islice(count(0, 2), 5)))
print(list(islice(cycle(SERVICES), 6)))
print(list(chain.from_iterable([[1, 2], [3], [4, 5]])))
print(list(islice((e.service for e in EVENTS if e.level == 'error'), 4)))
```

**Вывод.** `itertools` — замена вложенным циклам и буферным спискам: последовательности описываются декларативно и считаются лениво.

### 3.1. Ограничение и разбиение потоков

Бесконечный итератор становится полезным только с ограничителем: `islice` берёт отрезок, `takewhile` и `dropwhile` режут по предикату, `compress` фильтрует по маске-итератору. В отличие от срезов списка, `islice` не поддерживает шаг без полного перебора и умеет работать с `None` как правым краем.

Пакетирование (`batched`) — базовый приём при работе с потоками: элементы группируются по `n`, чтобы балансировать накладные расходы на одну операцию. Идиоматичная реализация — `iter(lambda: tuple(islice(it, n)), ())` или `zip(*[it] * n)`.

```python
from itertools import islice, compress, batched, takewhile

def batches(it, n):
    return list(iter(lambda: tuple(islice(it, n)), ()))

print([len(b) for b in batches(iter(EVENTS), 500)])

flags = [True, False, True, True]
print(list(compress(['auth', 'cart', 'billing', 'search'], flags)))

warm = takewhile(lambda e: e.level == 'info', EVENTS)
print(sum(1 for _ in warm), 'событий info подряд с начала')
```

**Вывод.** Над бесконечным источником всегда ставьте ограничитель; разбиение на батчи — способ сделать потоковую обработку пакетной.

### 3.2. `groupby` и требование отсортированности

`groupby` группирует только подряд идущие элементы с одинаковым ключом. Если последовательность не отсортирована по этому же ключу, одна логическая группа распадётся на несколько — это самое частое недоразумение модуля.

Значит, перед группировкой обязателен `sorted(iterable, key=...)`, а внутри цикла группа — тоже итератор: её нужно исчерпать до перехода к следующей группе, иначе данные будут потеряны. Для агрегатов по группам удобнее `defaultdict(list)` или pandas.

```python
from itertools import groupby

by_service = sorted(EVENTS, key=lambda e: e.service)
summary = {svc: (len(list(g)), round(sum(e.latency_ms for e in g) / 1, 1))
           for svc, g in groupby(by_service, key=lambda e: e.service)}
for svc in sorted(summary):
    print(svc, summary[svc])
```

**Вывод.** `groupby` — оператор над упорядоченным потоком: сортировка по ключу является частью контракта, а не деталями реализации.

### 4. Паттерны проектирования на средствах Python

Паттерн — имя для повторяющегося решения проекта, а не класс из книжного каталога. В Python часть структурных паттернов схлопывается до языка: вместо адаптера — утиная типизация, вместо декоратора-класса — обычный декоратор, вместо реестра фабрик — словарь функций.

В этой работе пять паттернов: **Фасад** — единая точка входа в подсистему; **Наблюдатель** — подписка на события вместо прямых вызовов; **Фабрика** — инкапсуляция создания объекта; **Стратегия** — алгоритм как передаваемое значение; **Синглтон** — ровно один экземпляр ресурса. Первые четыре опираются на функции-объекты и декораторы, пятый — на замыкание или класс с `__new__`.

Рис. 3. Пять паттернов работы и средства Python, которые реализуют их без классов-обёрток

```mermaid
flowchart LR
    P["паттерн GoF"] --> F["Фасад: класс-оркестратор"]
    P --> O["Наблюдатель: реестр коллбэков"]
    P --> B["Фабрика: словарь классов"]
    P --> S["Стратегия: словарь функций"]
    P --> N["Синглтон: __new__ или замыкание"]
    F --> C["опора: функции как значения, декораторы, словари"]
    O --> C
    B --> C
    S --> C
    N --> C
```



| Паттерн | Решает | Реализация в Python | Риск |
| --- | --- | --- | --- |
| Фасад | сложность подсистемы | класс-оркестратор | станется «богом» |
| Наблюдатель | связность модулей | реестр коллбэков | утечка подписок |
| Фабрика | создание сложного объекта | реестр классов | дублирование ветвлений |
| Стратегия | разветвлённая логика выбора | словарь функций | нет типов в ключе |
| Синглтон | единственный ресурс | `__new__` или замыкание | скрытое глобальное состояние |

**Вывод.** Паттерны в этой работе реализуются не «как в Java», а средствами языка: словарь функций, декоратор регистрации, ленивый модуль.

### 4.1. Фасад и Наблюдатель

**Фасад** прячет последовательность вызовов за одним методом. Он не добавляет логики, а ограничивает поверхность интеграции: вызывающий код знает один объект, а не пять классов внутри. Фасад — удобная граница для тестов и для соблюдения SRP.

**Наблюдатель** развязывает производителя и потребителей: подписка `on(event, callback)` и рассылка `notify(event, payload)`. Порядок уведомлений не гарантируется, поэтому обработчики обязаны быть независимыми и не бросать исключений наружу — иначе один сбойный подписчик остановит всю рассылку.

```python
class EventBus:
    def __init__(self):
        self._subs = {}

    def subscribe(self, topic, cb):
        self._subs.setdefault(topic, []).append(cb)

    def notify(self, topic, payload):
        for cb in self._subs.get(topic, []):
            try:
                cb(payload)
            except Exception as exc:      # сбойный подписчик не ломает рассылку
                print('subscriber failed:', exc)

class AlertingFacade:
    def __init__(self, bus, threshold):
        self.bus, self.threshold = bus, threshold

    def run(self, events):
        bad = [e for e in events if e.latency_ms > self.threshold]
        self.bus.notify('slow', bad)
        return len(bad)

bus = EventBus()
bus.subscribe('slow', lambda rows: print('alert:', len(rows), 'событий'))
print('обработано:', AlertingFacade(bus, 120.0).run(EVENTS))
```

**Вывод.** Фасад — про удобство вызова, Наблюдатель — про отсутствие прямой связанности; вместе они дают расширяемую подсистему без редактирования кода.

### 4.2. Фабрика, Стратегия и Синглтон

**Фабрика** переносит решение «какой класс создать» из вызывающего кода в один узел. Идиоматичная реализация — реестр классов и декоратор `@register`, который кладёт класс в словарь: добавление нового типа не требует правки ветвлений.

**Стратегия** параметризует алгоритм: вместо наследования и `if/elif` передаётся функция (или кортеж функций). Проверка — добавление новой стратегии без изменения контекста. Это и есть Open/Closed на практике.

**Синглтон** гарантирует один экземпляр: `__new__` с кэшем экземпляра, замыкание или модуль уровня. Цена — скрытая глобальная состояние и усложнение тестов, поэтому синглтон допустим только для действительно уникального ресурса (пул соединений, конфигурация) и всегда сопровождается способом сброса в тестах.

```python
AGGREGATORS = {}

def aggregator(name):
    def deco(func):
        AGGREGATORS[name] = func        # Фабрика + Стратегия: реестр функций
        return func
    return deco

@aggregator('max')
def agg_max(rows):
    return max(r.latency_ms for r in rows)

@aggregator('p50')
def agg_p50(rows):
    s = sorted(r.latency_ms for r in rows)
    return s[len(s) // 2]

def aggregate(rows, strategy):
    return AGGREGATORS[strategy](rows)

print(sorted(AGGREGATORS), aggregate(EVENTS[:100], 'p50'))

class Config:
    _instance = None

    def __new__(cls, *a, **kw):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, window_s=60):
        if not hasattr(self, 'window_s'):
            self.window_s = window_s

print(Config(30) is Config(999), Config().window_s)
```

**Вывод.** Реестр функций заменяет и Фабрику, и Стратегию; Синглтон реализуют через `__new__` или модуль — но только там, где единственный экземпляр оправдан.

---
## Примеры

Примеры ниже выполняются без изменений. Все они работают с потоком событий мониторинга — тем же доменом, что и задания 11–20.

### Пример 1. Декоратор измерения времени с сохранением метаданных

Обёртка замеряет время по `perf_counter` и печатает результат. `@wraps` сохраняет имя и docstring, поэтому функцию по-прежнему видно в профайлере под её собственным именем.

In [ ]:
from functools import wraps
from time import perf_counter

def measured(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        t0 = perf_counter()
        result = func(*args, **kwargs)
        took = (perf_counter() - t0) * 1000
        wrapper.last_ms = round(took, 3)
        print(f'{func.__name__}: ' + ('медленно' if took > 50 else 'в норме'))
        return result
    wrapper.last_ms = 0.0
    return wrapper

@measured
def slow_sum(rows):
    """Сумма задержек без использования builtins."""
    acc = 0.0
    for r in rows:
        acc += r.latency_ms
    return round(acc, 2)

print(slow_sum(EVENTS))
print(slow_sum.__name__, '|', slow_sum.__doc__)

### Пример 2. Мемоизация дорогой функции через `lru_cache`

Чистая функция подсчёта статистики по кортежу-ключу вызывается повторно. Кэш аргументов нельзя собрать для изменяемого списка, поэтому вход предварительно приводят к хешируемому виду.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=128)
def percentile(sorted_values, q):
    if not sorted_values:
        return 0.0
    idx = min(len(sorted_values) - 1, int(len(sorted_values) * q))
    return round(sorted_values[idx], 2)

lat = tuple(sorted(round(e.latency_ms, 2) for e in EVENTS))
for part in (0.5, 0.95, 0.99, 0.5):
    print(part, percentile(lat, part))
print(percentile.cache_info())

### Пример 3. Ленивый конвейер из генераторов

Фильтр, обогащение и агрегация соединены в цепочку генераторов: ни один промежуточный список не создаётся, а остановка на `islice` обходит только начало потока.

In [ ]:
from itertools import islice

def only_problems(rows):
    for r in rows:
        if r.level in ('warn', 'error'):
            yield r

def to_metric(rows):
    for r in rows:
        yield f'{r.service}:{r.level}', r.latency_ms

pipe = to_metric(only_problems(EVENTS))
first5 = list(islice(pipe, 5))
print(first5)
print('остаток после 5:', len(list(pipe)))

### Пример 4. Пакетирование потока и групповая агрегация

Поток разбивается на батчи по 250 событий, каждый батч группируется по уровню; `groupby` вызывается после сортировки по ключу — иначе группы перемешались бы.

In [ ]:
from itertools import islice, groupby

def batches(rows, n):
    return list(iter(lambda: tuple(islice(rows, n)), ()))

it = iter(EVENTS)
sizes = [len(b) for b in batches(it, 250)]
print('батчи:', sizes)

chunk = EVENTS[:250]
for level, g in groupby(sorted(chunk, key=lambda e: e.level),
                          key=lambda e: e.level):
    rows = list(g)
    print(level, len(rows), round(sum(r.latency_ms for r in rows) / len(rows), 1))

### Пример 5. Реестр стратегий и фасад поверх него

Декоратор регистрирует функцию-стратегию в словарь; фасад выбирает стратегию по имени и подписывает результат в шину событий. Новая стратегия добавляется без правки фасада — иллюстрация Open/Closed.

In [ ]:
REGISTRY = {}

def strategy(name):
    def deco(func):
        REGISTRY[name] = func
        return func
    return deco

@strategy('worst')
def worst(rows):
    return max(rows, key=lambda r: r.latency_ms)

@strategy('count')
def count(rows):
    return len(rows)

class Monitor:
    def __init__(self, threshold):
        self.threshold = threshold

    def report(self, rows, strategy_name):
        picked = [r for r in rows if r.latency_ms > self.threshold]
        return REGISTRY[strategy_name](picked)

m = Monitor(150.0)
res = m.report(EVENTS, 'worst')
print(sorted(REGISTRY), res.service, res.level, res.latency_ms)
print('событий выше порога:', m.report(EVENTS, 'count'))

---
## Часть II. Задания

### Задания на освоение концепции (1–10)

*Примеры решений отсутствуют. Студент решает самостоятельно.*

- Аннотации типов и `docstring` в каждом решении.
- Обработка ошибок там, где это уместно.
- Соблюдение PEP 8.

**Задание 1. Функция высшего порядка**

Реализуйте `apply_twice(func, value)`, применяющую `func` к значению дважды, и `compose(f, g) -> Callable`, возвращающую функцию `f(g(x))`. Проверьте композицию на парах `(abs, lambda x: x - 10)` и `(str.upper, str.strip)`.

In [ ]:
from typing import Callable

def apply_twice(func: Callable[[float], float], value: float) -> float:
    # Ваш код
    ...

def compose(f: Callable, g: Callable) -> Callable:
    # Ваш код
    ...

# Проверка
# print(apply_twice(lambda x: x * 2, 3))
# print(compose(abs, lambda x: x - 10)(4))

**Задание 2. Декоратор-счётчик вызовов**

Реализуйте `counted` без аргументов: обёртка увеличивает счётчик вызовов и печатает имя функции. Счётчик храните на объекте обёртки, метаданные сохраняйте через `@wraps`. Выведите `__name__`, `__doc__` и число вызовов после трёх запусков.

**Подсказка:** Атрибут обёртки доступен как `wrapper.calls`; инициализируйте его до возврата.

In [ ]:
from functools import wraps

def counted(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # Ваш код
        ...
    wrapper.calls = 0
    return wrapper

@counted
def scan(rows: list) -> int:
    """Счётчик отсканированных событий."""
    return len(rows)


**Задание 3. Декоратор с параметром: ограничение частоты**

Реализуйте `throttle(period_s)` — декоратор, который не позволяет вызывать функцию чаще, чем раз в `period_s`: при слишком раннем вызове возвращается последний результат без исполнения тела. Проверьте на функции-источнике времени.

In [ ]:
import time
from functools import wraps

def throttle(period_s: float):
    def deco(func):
        state = {'last': 0.0, 'result': None}

        @wraps(func)
        def wrapper(*args, **kwargs):
            # Ваш код
            ...
        return wrapper
    return deco

@throttle(0.05)
def now() -> float:
    return time.perf_counter()


**Задание 4. Класс-декоратор и условное декорирование**

Реализуйте класс-декоратор `Retry(times, exceptions)` с `__call__`, повторяющий вызов при указанном типе исключения и пробрасывающий его после исчерпания попыток. Затем напишите `maybe_deco(flag)`, который при `flag=False` возвращает функцию без изменений. Продемонстрируйте оба поведения на функции, падающей `ValueError`.

In [ ]:
class Retry:
    def __init__(self, func=None, times: int = 3, exceptions=(Exception,)):
        # Ваш код
        ...

    def __call__(self, *args, **kwargs):
        # Ваш код: поддержка и @Retry, и @Retry(times=5)
        ...

def maybe_deco(flag: bool):
    # Ваш код
    ...


**Задание 5. Декоратор валидации аргументов**

Реализуйте `@validate`, сверяющий аннотации параметров с фактическими типами аргументов и выбрасывающий `TypeError` с указанием имени параметра. Используйте `typing.get_type_hints` и `inspect.signature`. Проверьте на `f(x: int, tag: str)`.

In [ ]:
import inspect
from functools import wraps
from typing import get_type_hints

def validate(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # Ваш код
        ...
    return wrapper

@validate
def bucket(latency_ms: int, service: str) -> str:
    return f'{service}:{latency_ms}'


**Задание 6. Порядок применения декораторов**

Определите функцию с двумя декораторами, каждый из которых добавляет к результату префикс имени декоратора. Объясните в комментариях, почему получается именно такая строка, и покажите, что изменится при перестановке декораторов.

In [ ]:
def prefix(tag):
    def deco(func):
        def wrapper(*a, **kw):
            return tag + func(*a, **kw)
        return wrapper
    return deco

# Ваш код: два @prefix и тело функции
# print(...)


**Задание 7. Свой итератор по протоколу**

Реализуйте класс `DailyEvents(rows, day)`, итерируемый по событиям указанного дня через `__iter__` и `__next__` с корректным `StopIteration`. Добавьте поддержку `for` дважды подряд — повторный проход должен работать.

In [ ]:
class DailyEvents:
    def __init__(self, rows: list, day):
        # Ваш код
        ...

    def __iter__(self):
        # Ваш код
        ...

    def __next__(self):
        # Ваш код
        ...


**Задание 8. Генератор вместо списка**

Реализуйте `chunks(rows, n)` на `yield`, `flat_rows(groups)` на `yield from` и сравните пиковое потребление памяти и время для 200 000 событий в двух вариантах: список списков против генератора. Используйте `tracemalloc`.

In [ ]:
from typing import Iterable, Iterator

def chunks(rows: list, n: int) -> Iterator[list]:
    # Ваш код
    ...

def flat_rows(groups: Iterable[list]) -> Iterator[object]:
    # Ваш код через yield from
    ...


**Задание 9. Декоратор кэша по атрибуту экземпляра**

Реализуйте `@cached_result`, кэширующий результат метода в словаре экземпляра по ключу аргументов. Обеспечьте инвалидацию при изменении данных и покажите, что повторный вызов не пересчитывает агрегат.

In [ ]:
from functools import wraps

def cached_result(func):
    @wraps(func)
    def wrapper(self, *args):
        # Ваш код: self._cache
        ...
    return wrapper

class Stats:
    def __init__(self, rows):
        self.rows, self._cache = rows, {}

    @cached_result
    def total_latency(self, level: str) -> float:
        return round(sum(r.latency_ms for r in self.rows if r.level == level), 2)


**Задание 10. Инструменты itertools в одном запросе**

Одним выражением без промежуточных списков найдите: (а) 5 первых событий уровня `error`, (б) число событий в каждом из 4 батчей по 500, (в) все пары сервисов, встретившихся в одном батче. Используйте `islice`, `batched`, `combinations`.

In [ ]:
from itertools import islice, combinations

# (а)
# first5 = ...
# (б)
# per_batch = ...
# (в)
# pairs = ...


---
### Задания по прикладной задаче (11–20)

**Прикладная задача:** «Конвейер обработки событий мониторинга» — собрать подсистему, которая принимает поток событий, фильтрует и обогащает его ленивыми итераторами, измеряет и кэширует операции декораторами и собирает архитектуру из пяти базовых паттернов: Фасад, Наблюдатель, Фабрика, Стратегия, Синглтон

Решения опираются на единый набор данных — инициализация ниже.

### Инициализация данных для заданий 11–20

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import itertools, math, random, time

@dataclass(frozen=True)
class Event:
    ts: datetime
    service: str
    level: str          # 'info' | 'warn' | 'error'
    latency_ms: float
    payload: dict = field(default_factory=dict)

random.seed(42)
START = datetime(2026, 3, 1, 0, 0, 0)
SERVICES = ['auth', 'cart', 'billing', 'search']
LEVELS = ['info', 'info', 'info', 'warn', 'error']

EVENTS: list[Event] = [
    Event(START + timedelta(seconds=5 * i),
          random.choice(SERVICES),
          random.choice(LEVELS),
          round(random.lognormvariate(3.0, 0.9), 2))
    for i in range(2000)
]
print(len(EVENTS), EVENTS[0])


**Задание 11. Среда выполнения и доменная модель**

Инициализируйте репозиторий `python-labs`, папку `lab5` и ноутбук. Проверьте, что модуль инициализации данных создаёт ровно `len(EVENTS)` экземпляров `Event` с неизменяемыми полями, и убедитесь, что попытка изменить поле даёт ошибку.

In [ ]:
!git clone https://gitlab.com/<логин>/python-labs.git
%cd python-labs
!mkdir -p lab5
%cd lab5

# Проверка неизменяемости модели
# e = EVENTS[0]
# try:
#     e.latency_ms = 0.0
# except AttributeError as exc:
#     print('frozen:', type(exc).__name__)


**Задание 12. Ядро декораторов: `timed` и `counted`**

Реализуйте слой утилит `monitoring/decorators.py` (в ноутбуке — отдельная ячейка): `@timed(threshold_ms)`, `@counted`, `@retry(times, exceptions)` — все с `@wraps`. Приложите к каждой ячейку с тестом и печатным доказательством, что `__name__` и `__doc__` декорированной функции сохранились.

In [ ]:
# monitoring/decorators.py
def timed(threshold_ms: float = 100.0):
    ...

def counted(func):
    ...

def retry(times: int = 3, exceptions=(Exception,)):
    ...


**Задание 13. Ленивый источник событий**

Опишите источник `stream_events(rows, since=None, levels=None)` — генератор, отсекающий события по времени и уровню и ничего не буферизующий. Покажите, что выборка 10 событий из 2000 выполняется заметно быстрее полной фильтрации, и замерьте это декоратором `@timed`.

In [ ]:
from typing import Iterator, Optional

def stream_events(rows: list, since=None, levels: Optional[tuple] = None
                  ) -> Iterator[Event]:
    # Ваш код
    ...


**Задание 14. Конвейер обогащения**

Соберите конвейер из трёх генераторов: `stream_events` → `with_minutes` (добавляет номер минуты окна) → `with_severity` (в словаре `SEVERITY` задаёт вес уровня). Все три должны работать лениво; выведите 3 первых кортежа и полную статистику по весам.

**Подсказка:** Номер минуты: `(e.ts - START).total_seconds() // 60`.

In [ ]:
SEVERITY = {'info': 1, 'warn': 3, 'error': 10}

def with_minutes(rows):
    ...

def with_severity(rows):
    ...

# pipe = with_severity(with_minutes(stream_events(EVENTS)))


**Задание 15. Батчи и скользящая статистика через `itertools`**

Разбейте поток на окна по 5 минут функциями `itertools` и для каждого окна посчитайте число ошибок, максимальную и среднюю задержку. Реализацию сделайте без явных вложенных циклов: `batched`/`islice` плюс генератор агрегатов.

In [ ]:
from itertools import batched, islice

def windows(rows, minutes=5):
    ...

def window_stats(win):
    ...

# for i, st in enumerate(islice((window_stats(w) for w in windows(EVENTS)), 6)):
#     print(i, st)


**Задание 16. Стратегии агрегации как значения**

Реализуйте реестр стратегий `agg_max`, `agg_p50`, `agg_error_ratio`, `agg_weighted` через декоратор `@strategy(name)` из теории. Функция `aggregate` должна принимать стратегию по имени и по вызываемому объекту. Добавьте новую стратегию, не изменяя `aggregate`, и зафиксируйте это в выводе.

In [ ]:
STRATEGIES = {}

def strategy(name: str):
    ...

def aggregate(rows, strategy_name):
    ...

# STRATEGIES.keys() -> ['agg_max', ...]


**Задание 17. Фабрика окон и Синглтон конфигурации**

Сделайте Фабрику `create_window(kind, size)`, возвращающую объект окна (`Tumbling` или `Sliding`) из реестра классов, и Синглтон `RuntimeConfig`, хранящий порог задержки и размер окна. Продемонстрируйте `is`-эквивалентность двух обращений к конфигу и возможность сброса состояния в тестах.

In [ ]:
class Window:
    def add(self, event):
        ...

class Tumbling(Window):
    ...

class Sliding(Window):
    ...

WINDOWS = {}

def create_window(kind: str, size: int) -> Window:
    ...

class RuntimeConfig:
    _instance = None

    @classmethod
    def get(cls):
        ...


**Задание 18. Наблюдатель: алерты по подписке**

Реализуйте `EventBus` с `subscribe(topic, cb)` и `notify(topic, payload)` так, чтобы исключение подписчика не прерывало рассылку и попадало в лог. Подпишите обработчики «письмо», «веб-хук» и «метрика» на тему `breach` и сгенерируйте алерты для всех окон из задания 15.

**Подсказка:** Возвращайте число успешно отработавших подписчиков — это упростит проверку.

In [ ]:
import logging

class EventBus:
    def __init__(self):
        self._subs = {}

    def subscribe(self, topic: str, cb) -> None:
        ...

    def notify(self, topic: str, payload) -> int:
        ...


**Задание 19. Фасад `MonitoringPipeline`**

Соберите фасад, принимающий сырые события и конфигурацию и предоставляющий один метод `process(rows) -> Report`: ленивый поток, окна, стратегии, алерты через шину. Фасад не должен знать имена конкретных стратегий — они приходят из конфигурации. Покажите вызов фасада с двумя разными наборами стратегий.

In [ ]:
class Report:
    def __init__(self, rows, breaches, total_latency):
        ...

class MonitoringPipeline:
    def __init__(self, config, bus, strategies):
        ...

    def process(self, rows) -> Report:
        ...

    def to_text(self, report) -> str:
        ...


**Задание 20. Обоснование архитектуры и сдача**

Составьте `README.md`: таблица «паттерн — где реализован — какую проблему решает», обоснование применения каждого из пяти паттернов, ссылка на репозиторий. Отметьте, где использованы функции первого класса и декораторы регистрации. Проверьте выполнение ноутбука сверху вниз, сделайте коммит и `git push`.

In [ ]:
%%writefile README.md
# Лабораторная работа № 5

## Тема
Декораторы, генераторы и itertools. Базовые паттерны проектирования.

## Паттерны
| Паттерн | Реализация | Проблема |
|---|---|---|
| Фасад | MonitoringPipeline | ... |

## Ссылка
https://gitlab.com/<логин>/python-labs


## Критерии оценки

- Декораторы сохраняют метаданные через `@wraps` и не ломают сигнатуру оригинала.
- Конвейер обработки событий ленив: промежуточные списки не материализуются в заданиях 13–15.
- Все пять паттернов реализованы средствами Python и названы в README с обоснованием.
- Новая стратегия или новый подписчик добавляются без правки фасада (соблюдён OCP).
- Обработка ошибок в шине событий: сбойный подписчик не останавливает рассылку.
- Код оформлен по PEP 8, содержит аннотации типов и docstrings, выполняется без ошибок.

---
## Контрольные вопросы

1. Что происходит с именем и docstring функции, если декоратор не использует `@wraps`?
2. Чем `@deco` эквивалентен в явном виде и когда тело декоратора выполняется — один раз или при каждом вызове?
3. Как устроен декоратор с аргументами и почему для него нужно три уровня вложенности?
4. В каком порядке применяются несколько декораторов и в каком порядке исполняются обёртки при вызове?
5. Чем итерируемый объект отличается от итератора и что выбрасывает `__next__` при исчерпании?
6. Почему генератор можно пройти только один раз и как это влияет на проектирование конвейеров?
7. Зачем `groupby` требует предварительной сортировки и что произойдёт без неё?
8. Какие три группы инструментов содержит `itertools` и чем опасны бесконечные итераторы?
9. Чем Фабрика отличается от Стратегии и как обе реализуются словарём и декоратором регистрации?
10. Почему Синглтон считается антипаттерном в большинстве случаев и в каком единственном случае он оправдан?

---
## Дополнительно

1. Сохраните ноутбук
   Имя файла `lab5_decorators_itertools.ipynb`; проверьте выполнение Kernel → Restart & Run All.
2. Подключите репозиторий
   В репозитории `python-labs` создайте папку `lab5` и поместите в неё ноутбук и `README.md`.
3. Добавьте `.gitignore`
   Исключите `__pycache__/`, `*.pyc`, `.ipynb_checkpoints/`, `.env`.
4. Коммит и push
5. Прикрепите результат
   Ссылку на публичный репозиторий и файл ноутбука загрузите в систему дистанционного обучения.

In [ ]:
!git add .
!git commit -m "ЛР5: декораторы, itertools, паттерны"
!git push origin main